<a href="https://colab.research.google.com/github/kmouts/LazySlide/blob/main/tutorials/skip/multiple_slides.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Integration with slide-level labels

In this tutorial we will demonstrate how to integrate whole slide images (WSIs) with slide-level labels and derive quantitative scores for each slide via top-K scoring.

We will also demonstrate how to run tasks in a distributed fashion using `dask`.

For this, we will be using a pre-processed dataset of artery tissue from GTEx, which contains healthy and calcified samples.

In [1]:
from huggingface_hub import hf_hub_download
import pandas as pd

table = hf_hub_download(
    "rendeirolab/lazyslide-data", "GTEx_artery_dataset.csv.gz", repo_type="dataset"
)

dataset = pd.read_csv(table)
dataset.head()

GTEx_artery_dataset.csv.gz:   0%|          | 0.00/510 [00:00<?, ?B/s]

,Tissue Sample Id,Sex,Age Bracket,Pathology Categories
0,GTEX-111YS-2226,male,60-69,calcification
1,GTEX-11GSP-2926,female,60-69,calcification
2,GTEX-11LCK-1426,male,30-39,clean_specimens
3,GTEX-11ONC-2726,male,60-69,calcification
4,GTEX-12126-0726,male,20-29,clean_specimens


Here I prepared a set of terms to perform semantic analysis on WSIs. Using a curated set of semantic terms helps make the output of vision-text models more consistent and easier to interpret.

In [2]:
terms = [
    "BMP-2",
    "Monckeberg sclerosis",
    "Runx2",
    "adventitia",
    "apoptosis",
    "arterial hardening",
    "arterial narrowing",
    "arterial remodeling",
    "arterial stiffness",
    "arteriole",
    "artery",
    "atherosclerosis",
    "basement membrane",
    "blood flow",
    "bone morphogenetic protein",
    "calcification",
    "calcified nodule",
    "calcium deposition",
    "calcium phosphate",
    "chronic kidney disease",
    "collagen",
    "compliance",
    "connective tissue",
    "elastic fibers",
    "elasticity",
    "endothelial dysfunction",
    "endothelium",
    "epithelium",
    "external elastic lamina",
    "extracellular matrix",
    "fibroblast",
    "fibrosis",
    "fibrous cap",
    "gap junction",
    "hemodynamics",
    "hydroxyapatite",
    "hyperphosphatemia",
    "inflammation",
    "internal elastic lamina",
    "interstitial space",
    "intima",
    "intimal calcification",
    "intimal thickening",
    "ischemia",
    "lamina propria",
    "lumen",
    "macrocalcification",
    "macrophage",
    "matrix vesicle",
    "mechanotransduction",
    "media",
    "medial calcification",
    "microcalcification",
    "mineralization",
    "myofibroblast",
    "necrotic core",
    "osteoblast-like cell",
    "osteocalcin",
    "osteogenic",
    "osteopontin",
    "oxidative stress",
    "pericyte",
    "phosphate transporter",
    "plaque",
    "shear stress",
    "smooth muscle",
    "tight junction",
    "tunica",
    "vasa vasorum",
    "vascular basement membrane",
    "vascular compliance",
    "vascular integrity",
    "vascular niche",
    "vascular ossification",
    "vascular remodeling",
    "vascular smooth muscle cell",
    "vascular stiffness",
    "vascular tone",
    "vascular wall",
]

Since we need to run for many slides, let's first define a function to process a slide and reuse it.

In [3]:
!pip install wsidata
import sys
if 'lazyslide' not in sys.modules:
  !pip install lazyslide

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 kB 5.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of ome-zarr to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.8/87.8 kB 10.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [26]:
!pip install dask-jobqueue

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.7 MB/s eta 0:00:00


In [16]:
from wsidata import open_wsi
import lazyslide as zs
from pathlib import Path

def wsi_feature_extraction(slide):
    print(f"Processing slide: {slide}")
    s = hf_hub_download(
        "rendeirolab/lazyslide-data",
        f"gtex_artery_data/{slide}.svs",
        repo_type="dataset",
    )
    # Define the output directory and create it if it doesn't exist
    output_dir = Path("data")
    output_dir.mkdir(parents=True, exist_ok=True)

    # Construct the full Zarr file path for this slide
    output_path = output_dir / f"{slide}.zarr"

    # Pass the full Zarr file path to the store parameter of open_wsi
    wsi = open_wsi(s, attach_thumbnail=False, store=output_path)

    zs.pp.find_tissues(wsi)
    zs.pp.tile_tissues(wsi, 256, mpp=0.5, background_fraction=0.5)

    # conch feature
    zs.tl.feature_extraction(wsi, "conch", pbar=False)
    zs.tl.feature_aggregation(wsi, "conch")
    embed = zs.tl.text_embedding(terms, "conch")
    zs.tl.text_image_similarity(wsi, embed, "conch")

    try:
        print(f"Attempting to write Zarr file to {output_path}/")
        wsi.write()
        if output_path.exists():
            print(f"Successfully wrote Zarr file for {slide} at {output_path}/")
        else:
            print(f"Failed to write Zarr file for {slide} at {output_path}/: File does not exist after write.")
    except Exception as e:
        print(f"Error writing Zarr file for {slide}: {e}")


In [17]:
# Test with a single slide outside of Dask to debug
single_slide_id = dataset['Tissue Sample Id'].iloc[0] # Get the first slide ID
wsi_feature_extraction(single_slide_id)

# Verify if the file was created
output_path = Path(f"data/{single_slide_id}.zarr")
if output_path.exists():
    print(f"Successfully created {output_path}/ locally.")
else:
    print(f"Failed to create {output_path}/ locally. Check the 'data/' directory and write permissions.")

Processing slide: GTEX-111YS-2226


/usr/local/lib/python3.12/dist-packages/lazyslide_models/multimodal/conch.py:27: UserWarning: As from v0.8.2, Normalization will not be applied to image embedding of CONCH model anymore.A `normalize=True` argument is added to the `text_image_similarity` method.If you only use the image embedding for text image similarity, you can safely ignore this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lazyslide_models/multimodal/conch.py:27: UserWarning: As from v0.8.2, Normalization will not be applied to image embedding of CONCH model anymore.A `normalize=True` argument is added to the `text_image_similarity` method.If you only use the image embedding for text image similarity, you can safely ignore this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lazyslide/tools/_text_annotate.py:162: UserWarning: As of v0.8.2, the image embedding from image text model is not normalized after feature extraction by default. The normalization is applied here (text_ima

Attempting to write Zarr file to data/GTEX-111YS-2226.zarr/


/usr/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/usr/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/usr/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/usr/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/usr/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/usr/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/usr/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(se

Successfully wrote Zarr file for GTEX-111YS-2226 at data/GTEX-111YS-2226.zarr/
Successfully created data/GTEX-111YS-2226.zarr/ locally.


## Run for all slides

The easiest way is to run a for-loop:

```python
for slide in dataset["Tissue Sample Id"]:
    wsi_feature_extraction(slide)
```

However, this will take a long time and doesn't fully use the power of parallelization.

In [6]:
!pip install git+https://github.com/mahmoodlab/CONCH.git

  Cloning https://github.com/mahmoodlab/CONCH.git to /tmp/pip-req-build-8ehg6qfy
  Running command git clone --filter=blob:none --quiet https://github.com/mahmoodlab/CONCH.git /tmp/pip-req-build-8ehg6qfy
  Resolved https://github.com/mahmoodlab/CONCH.git to commit 141cc09c7d4ff33d8eda562bd75169b457f71a62
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00
  Created wheel for conch: filename=conch-0.1.0-py3-none-any.whl size=418185 sha256=70bc2442c37f825cd301408b956c0a9525af075f65cff4cc18621da25b47d708
  Stored in directory: /tmp/pip-ephem-wheel-cache-yikc9oja/wheels/2e/12/b4/2b640007ff2067a570f11f7edddbfae423e6aba2b7ba492b3c
Successfully built conch


In [8]:
for slide in dataset["Tissue Sample Id"]:
    wsi_feature_extraction(slide)

/usr/local/lib/python3.12/dist-packages/lazyslide_models/multimodal/conch.py:27: UserWarning: As from v0.8.2, Normalization will not be applied to image embedding of CONCH model anymore.A `normalize=True` argument is added to the `text_image_similarity` method.If you only use the image embedding for text image similarity, you can safely ignore this warning.
  warnings.warn(


meta.yaml:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/802M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/lazyslide_models/multimodal/conch.py:27: UserWarning: As from v0.8.2, Normalization will not be applied to image embedding of CONCH model anymore.A `normalize=True` argument is added to the `text_image_similarity` method.If you only use the image embedding for text image similarity, you can safely ignore this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lazyslide/tools/_text_annotate.py:162: UserWarning: As of v0.8.2, the image embedding from image text model is not normalized after feature extraction by default. The normalization is applied here (text_image_similarity),if your features are extracted in previous versions, consider setting normalize=False.
  warnings.warn(msg, stacklevel=find_stack_level())
/usr/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
/usr/lib/python3.12/contextlib.py:137: UserWarning: zarr v3 autosharding will b

gtex_artery_data/GTEX-11GSP-2926.svs:   0%|          | 0.00/134M [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/lazyslide_models/multimodal/conch.py:27: UserWarning: As from v0.8.2, Normalization will not be applied to image embedding of CONCH model anymore.A `normalize=True` argument is added to the `text_image_similarity` method.If you only use the image embedding for text image similarity, you can safely ignore this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lazyslide_models/multimodal/conch.py:27: UserWarning: As from v0.8.2, Normalization will not be applied to image embedding of CONCH model anymore.A `normalize=True` argument is added to the `text_image_similarity` method.If you only use the image embedding for text image similarity, you can safely ignore this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lazyslide/tools/_text_annotate.py:162: UserWarning: As of v0.8.2, the image embedding from image text model is not normalized after feature extraction by default. The normalization is applied here (text_ima

gtex_artery_data/GTEX-ZVZP-2726.svs:   0%|          | 0.00/379M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/lazyslide_models/multimodal/conch.py:27: UserWarning: As from v0.8.2, Normalization will not be applied to image embedding of CONCH model anymore.A `normalize=True` argument is added to the `text_image_similarity` method.If you only use the image embedding for text image similarity, you can safely ignore this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lazyslide_models/multimodal/conch.py:27: UserWarning: As from v0.8.2, Normalization will not be applied to image embedding of CONCH model anymore.A `normalize=True` argument is added to the `text_image_similarity` method.If you only use the image embedding for text image similarity, you can safely ignore this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lazyslide/tools/_text_annotate.py:162: UserWarning: As of v0.8.2, the image embedding from image text model is not normalized after feature extraction by default. The normalization is applied here (text_ima

gtex_artery_data/GTEX-ZYT6-1526.svs:   0%|          | 0.00/30.4M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/lazyslide_models/multimodal/conch.py:27: UserWarning: As from v0.8.2, Normalization will not be applied to image embedding of CONCH model anymore.A `normalize=True` argument is added to the `text_image_similarity` method.If you only use the image embedding for text image similarity, you can safely ignore this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lazyslide_models/multimodal/conch.py:27: UserWarning: As from v0.8.2, Normalization will not be applied to image embedding of CONCH model anymore.A `normalize=True` argument is added to the `text_image_similarity` method.If you only use the image embedding for text image similarity, you can safely ignore this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lazyslide/tools/_text_annotate.py:162: UserWarning: As of v0.8.2, the image embedding from image text model is not normalized after feature extraction by default. The normalization is applied here (text_ima

## Distributed processing with dask

Dask is a good option for parallelization on local machine or across multiple machines.

For different hardware availabilities, alternatives are:
1. [dask-jobqueue](https://jobqueue.dask.org/en/latest/): For PBS, Slurm, MOAB, SGE, LSF, and HTCondor.
2. [coiled](https://docs.coiled.io/user_guide/index.html): AWS, GCP, Azure etc.
3. [dask-cuda](https://docs.rapids.ai/api/dask-cuda/nightly/quickstart/): If you have multiple GPU cards locally.

Here, we showcase how to parallel the jobs with dask on a SLURM cluster.
The configuration may not work on your SLURM system, please make adjustment accordingly.

When running GPU-intensive work like feature extraction for multiple WSIs,
we recommend to run one task on one GPU every time.
To accelerate the processing speed, either distribute across multiple GPU cards or multiple machines.

Here are code snippet to run on different architectures

Run local with CPUs:

```python
from dask.distributed import LocalCluster
cluster = LocalCluster()
```

Run local with many GPUs:

```python
from dask_cuda import LocalCUDACluster
cluster = LocalCUDACluster()
```

Run on a SLURM cluster with GPUs (Example script, may not work on users' cluster):

```python
from dask_jobqueue import SLURMCluster

cluster = SLURMCluster(
    queue="gpu",
    cores=8,
    processes=1,
    memory="20 GB",
    # For SLURM, use --gres flag to get GPU
    job_extra_directives=["--gres=gpu:h100pcie:1"],
    # Each work must one GPU
    worker_extra_args=["--resources GPU=1"],
)
```

In [31]:
from dask_jobqueue import SLURMCluster

cluster = SLURMCluster(
    queue="gpu",
    cores=8,
    processes=1,
    memory="20 GB",
    interface="eth0", # Changed from 'ib1' to 'eth0'
    job_extra_directives=["-q gpu", "--gres=gpu:l4_gpu:1", "--time=2:00:00"],
    worker_extra_args=["--resources GPU=1"],
    log_directory="./dask-logs",
)

INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:   tcp://172.28.0.12:44347
INFO:distributed.scheduler:  dashboard at:  http://172.28.0.12:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle


In [32]:
from dask.distributed import Client

client = Client(cluster)
cluster.adapt(minimum=1, maximum=10)

INFO:distributed.scheduler:Receive client connection: Client-9c07841c-6cbb-11f1-a0e9-0242ac1c000c
INFO:distributed.core:Starting established connection to tcp://172.28.0.12:50472
INFO:distributed.deploy.adaptive:Adaptive scaling started: minimum=1 maximum=10


In [ ]:
client

Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: http://10.110.89.41:8787/status,
Dashboard: http://10.110.89.41:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.110.89.41:36261,Workers: 0
Dashboard: http://10.110.89.41:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


Let's parallelize the jobs

In [33]:
futures = [
    client.submit(wsi_feature_extraction, slide, resources={"GPU": 1})
    for slide in dataset["Tissue Sample Id"]
]

If you want to monitor the process, you can either go to the dask dashboard or use a simple progress bar

In [ ]:
from dask.distributed import as_completed
from tqdm.auto import tqdm

for _ in tqdm(as_completed(futures), total=len(futures)):
    pass

  0%|          | 0/45 [00:00<?, ?it/s]

In [ ]:
client.shutdown()

We can calculate the scores for all pathological terms that we defined and save them for further analysis.

In [9]:
from pathlib import Path
from anndata import read_zarr

slide_scores = {}
for store in Path("data").glob("*.zarr"):
    adata = read_zarr(store / "tables" / "conch_tiles_text_similarity")
    scores = zs.metrics.topk_score(adata, k=100)
    slide_scores[store.stem] = dict(zip(adata.var.index, scores))

In [10]:
slide_scores = pd.DataFrame(slide_scores).T

In [11]:
num_slides = dataset['Tissue Sample Id'].nunique()
print(f"The gtex_artery_data dataset has {num_slides} slides.")

The gtex_artery_data dataset has 45 slides.


## Slide aggregation

After the slides are processed to have slide-level features and scores, we will aggregate them into an AnnData object.

In [12]:
from wsidata import agg_wsi

dataset["store"] = [f"data/{s}.zarr" for s in dataset["Tissue Sample Id"]]
agg_data = agg_wsi(dataset, "conch", store_col="store", agg_key="agg_slide")
agg_data.obs = agg_data.obs.join(slide_scores, on="Tissue Sample Id")
agg_data

ValueError: File data/GTEX-WHSE-1126.zarr not existed.

In [ ]:
agg_data.write_h5ad("agg_conch_features.h5ad")

In [13]:
from pathlib import Path

expected_zarr_files = {f"data/{s}.zarr" for s in dataset['Tissue Sample Id']}
actual_zarr_files = {str(p) for p in Path("data").glob("*.zarr")}

missing_files = expected_zarr_files - actual_zarr_files

if missing_files:
    print(f"The following {len(missing_files)} Zarr files are missing from the 'data/' directory:")
    for f in sorted(list(missing_files)): # Sort for consistent output
        print(f)
else:
    print("All expected Zarr files are present in the 'data/' directory.")


The following 45 Zarr files are missing from the 'data/' directory:
data/GTEX-111YS-2226.zarr
data/GTEX-11GSP-2926.zarr
data/GTEX-11LCK-1426.zarr
data/GTEX-11ONC-2726.zarr
data/GTEX-12126-0726.zarr
data/GTEX-12KS4-1226.zarr
data/GTEX-1339X-2626.zarr
data/GTEX-13O61-2526.zarr
data/GTEX-13RTK-1826.zarr
data/GTEX-13W46-0626.zarr
data/GTEX-144GM-2226.zarr
data/GTEX-14BMU-2426.zarr
data/GTEX-14ICL-1826.zarr
data/GTEX-14PJN-2226.zarr
data/GTEX-15RJE-0526.zarr
data/GTEX-16MT9-0326.zarr
data/GTEX-16NPV-0526.zarr
data/GTEX-16YQH-0626.zarr
data/GTEX-17MF6-0526.zarr
data/GTEX-183FY-0526.zarr
data/GTEX-1C6VS-0326.zarr
data/GTEX-1CAMQ-0426.zarr
data/GTEX-1H23P-0426.zarr
data/GTEX-1HGF4-2826.zarr
data/GTEX-1HUB1-0226.zarr
data/GTEX-1I1GU-1926.zarr
data/GTEX-1NV5F-0426.zarr
data/GTEX-O5YU-0926.zarr
data/GTEX-PW2O-1926.zarr
data/GTEX-S33H-2426.zarr
data/GTEX-S4Q7-1326.zarr
data/GTEX-SNMC-1526.zarr
data/GTEX-T5JW-2126.zarr
data/GTEX-TKQ1-1226.zarr
data/GTEX-U8XE-0626.zarr
data/GTEX-WHSE-1126.zarr
data/